## Multimodal RAG with GPT4V and LangChain

#### When do we need Multimodal RAG
Standard RAG is easy with text-only files, but what if we want to use RAG with pdfs or slides that have text, images, and tables? Then we use Multimodal RAG.

#### Multimodal RAG explained
* Summarize text with the LLM model.
* Summarize table with the LLM model.
* Summarize images with the new Multimodal LLM model (GPT4V).
* Convert summaries into numbers (embeddings) and store the embeddings in a multivector retriever (vector database).
* Store the raw documents (the text and the summary of the images) in a DocumentStore.
* When a question is asked, do similarity search to retrieve the most relevant docs and send the response to the LLM Model to format it properly using natural language.

In [1]:
#!pip install python-dotenv

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

api_key = os.environ["OPENROUTER_API_KEY"]

#### Install LangChain

If you are using the pre-loaded poetry shell, you do not need to install the following package because it is already pre-loaded for you:

In [3]:
#!pip install langchain

## Connect with an LLM and start a conversation with it

If you are using the pre-loaded poetry shell, you do not need to install the following packages because they are already pre-loaded for you:

In [4]:
#!pip install langchain-openai

In [5]:
#!pip install langchain-community

* For this project, we will use OpenAI's gpt-3.5-turbo and gpt4o. **The model gpt4-vision-preview has been deprecated by OpenAI and it is replaced now by the model gpt-4o**.

In [6]:

from langchain_openai import ChatOpenAI
''' 
chain_gpt_35 = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=1024)
chain_gpt_4_vision = ChatOpenAI(model="gpt-4o", max_tokens=1024)
'''


chain_gpt_35 = ChatOpenAI(
    model="openai/gpt-oss-120b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    max_tokens=1024
)

chain_gpt_4_vision = ChatOpenAI(
    model="llava-13b",
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    max_tokens=1024
)

## Multimodal RAG App

If you are using the pre-loaded poetry shell, you do not need to install the following packages because they are already pre-loaded for you:

In [7]:
#!pip install openai

In [8]:
#!pip install pydantic lxml tiktoken

In [9]:
#!pip install langchain-chroma

In [10]:
#!pip install "unstructured[all-docs]"

* The unstructured module is the key here. We will use it to extract all the relevant parts of the document (text, tables and images).
* Chromadb will be our vector store.

#### In order to use the unstructured module, we will need to install two other modules: tesseract and poppler
* In MacOS with Homebrew:
    * brew install tesseract
    * brew install poppler
* For other systems (Windows, etc):
    * [info on how to install tesseract](https://tesseract-ocr.github.io/tessdoc/Installation.html)
    * [info on how to install poppler](https://pdf2image.readthedocs.io/en/latest/installation.html)


video: https://youtu.be/HNCypVfeTdw?si=o545WJ0FfMcM5IcC

https://youtu.be/IDu46GjahDs?si=8WLl6Wfu8yqGT1CQ

#### We will use a fake startupai-financial-report-v2.pdf file with text, tables and images

#### What we will do next:
* Import partition_pdf from the unstructured package
* We set the tesseract_cmd to the path where we store our tesseract.exe file
* We set input_path and output_path
* We then create the raw_pdf_elements and run the partition_pdf function from the unstructured package:
    * we set the filename and path
    * we instruct unstructured to extract all the relevant parts of the file (text, tables and images)
    * we set chunking strategy
    * we set the output path
* The following cell can take a few seconds to run

In [14]:
from pypdf import PdfReader

reader = PdfReader("startupai-financial-report-v2.pdf")

for page in reader.pages:
    print(page.extract_text())

202020212022202320240510152025
Explore our financial performance through
balance sheets, income, and cash flow
statements.
StartupAI boasts an impressive return on
investment (ROI), demonstrating its financial
acumen and ability to generate substantial
profits for its stakeholders through strategic
decisions and operational excellence.
StartupAI has achieved a
remarkable $22 million in sales,
showcasing its market dominance
and strong customer appeal.
www.startupAI.com +123-456-7890DELAITTE
GROSS INCOME 22,000,000 $
TOTAL EXPENSES  2.000,000 $
TAXES  5,000.000 $
NET INCOME 15,000,000 $


In [19]:
from pypdf import PdfReader

reader = PdfReader("startupai-financial-report-v2.pdf")

all_text = ""
for page in reader.pages:
    all_text += page.extract_text()

## See what we have in the raw_pdf_elements variable
* The classes with CompositeElements are text
* The classes with Table are tables

In [20]:
response = chain_gpt_35.invoke(
    f"Summarize this document:\n\n{all_text[:3000]}"
)

print(response.content)

**Summary**

- **Financial Overview**  
  - The document highlights StartupAI’s key financial statements (balance sheet, income statement, cash‑flow statement).  

- **Performance Highlights**  
  - **Revenue:** $22 million in sales, indicating strong market presence.  
  - **Gross Income:** $22 million.  
  - **Total Expenses:** $2 million.  
  - **Taxes:** $5 million.  
  - **Net Income:** $15 million, reflecting a healthy profit margin.  

- **Return on Investment**  
  - StartupAI delivers a high ROI, showcasing its ability to turn strategic decisions into substantial stakeholder returns.  

- **Contact Information**  
  - Website: www.startupAI.com  
  - Phone: +1‑123‑456‑7890  
  - Auditor: Deloitte  

Overall, StartupAI demonstrates robust financial health, strong sales performance, and effective cost management, positioning it as a dominant player in its market.


## Now we want to extract the relevant information
* We want to store the text, table and image elements in 3 lists.
* We cannot send the images as they are, we need to convert them into binary format with base64.
* For the text and table elements we will loop to add them in their list.

In [21]:
from pypdf import PdfReader

reader = PdfReader("startupai-financial-report-v2.pdf")

texts = []
for page in reader.pages:
    texts.append(page.extract_text())

print("pages:", len(texts))

pages: 1


In [23]:
chain_gpt_35 = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

In [24]:
response = chain_gpt_35.invoke(
    f"Summarize:\n\n{texts[0]}"
)

print(response.content)

StartupAI’s financial snapshot shows strong profitability: it generated **$22 million in gross income (sales)**, incurred **$2 million in total expenses**, paid **$5 million in taxes**, and ended with a **net income of $15 million**. This translates to an impressive return on investment, reflecting strategic decision‑making and operational excellence. The company’s $22 million sales figure underscores its market dominance and customer appeal. For more details—including balance sheets, income statements, and cash‑flow reports—visit www.startupAI.com or call +123‑456‑7890.


#### Images
* They are currently stored in the "figures" folder.
* We will loop through that folder:
    * check if the image file ends with png, jpg, jpeg
    * then provide the full page to the encode_image function to encode it in a base64 format
    * and then enter the encoding result in the image list
* The following cell may take a few seconds to run. 

## Now we can create 3 functions to summarize the texts, table and images
* for the text and table the functions are very similar
* for the images we use GPT4V
* pay attention on how we set the url

In [30]:
from langchain.schema.messages import HumanMessage, AIMessage

# Function for text summaries
def summarize_text(text_element):
    prompt = f"Summarize the following text:\n\n{text_element}\n\nSummary:"
    response = chain_gpt_35.invoke([HumanMessage(content=prompt)])
    return response.content

# Function for table summaries
def summarize_table(table_element):
    prompt = f"Summarize the following table:\n\n{table_element}\n\nSummary:"
    response = chain_gpt_35.invoke([HumanMessage(content=prompt)])
    return response.content

# Function for image summaries
def summarize_image(encoded_image):
    prompt = [
        AIMessage(content="You are a bot that is good at analyzing images."),
        HumanMessage(content=[
            {
                "type": "text", 
                "text": "Describe the contents of this image."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{encoded_image}"
                },
            },
        ])
    ]
    response = chain_gpt_4_vision.invoke(prompt)
    return response.content

## Now we will create a summary for each text, table and image element
* The following cells will take some time to run.
* Careful: GPT4V is significantly more expensive than the regular GPT models.
* Note: If you try to summarize all them, the Jupyter Kernel may crash occassionally. It that happens, you will have to run it again. 

In [33]:
text_summaries = []

for i, text in enumerate(texts[:2]):   
    summary = chain_gpt_35.invoke(
        f"Summarize this:\n\n{text[:2000]}"
    )
    
    text_summaries.append(summary.content)
    print(f"{i + 1}th element processed.")

1th element processed.


In [34]:
table_summaries = []

for i, text in enumerate(texts[:1]):
    summary = chain_gpt_35.invoke(
        f"If there is a table in this text, explain it clearly:\n\n{text[:2000]}"
    )
    
    table_summaries.append(summary.content)
    print(f"{i + 1}th element processed.")

1th element processed.


## After creating the summaries, we can now proceed with the RAG technique
* We will use LangChain's [Multi-Vector Retriever](https://python.langchain.com/docs/modules/data_connection/retrievers/multi_vector) to store:
    * all our documents,
    * the summaries,
    * and also the embeddings
    * in a vector database
* We will use a Chroma vector database
* We will use a docstore to store the raw documents (the original documents). For that we will use InMemoryStore from LangChain.
* We will provide later the id_key, now it is just a string
* Then we create the function to add documents to the retriever
    * create some uuids for our documents using the uuid4() function.
        * The uuid4 function in Python is part of the uuid module, which generates unique identifiers according to the UUID (Universally Unique Identifier) standard.
        * The uuid4 function specifically generates a random UUID based on the version 4 specification. This means that each time you call uuid4, it generates a completely random UUID that is highly unlikely to be duplicated anywhere else, now or in the future.
        * A UUID generated by uuid4 looks something like this: 12345678-1234-5678-1234-567812345678, where each digit is a hexadecimal character (0-9, a-f), representing a 128-bit value.
        * The version 4 UUIDs are useful for situations where you need to ensure uniqueness across different systems without the need for a central coordinating mechanism.
    * then we will create a list of documents using the Document class
        * for each page_content, we include the summary of the element
        * for each metadata, we enter the uuid
    * Then we add the documents to the vector database
    * We also store our raw documents in the docstore. Each raw document has the corresponding uuid.
    * As you can see, the connection between the vector database and the docstore is in the uuids.

In [36]:
import uuid

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.schema.document import Document
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma

# free embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Vector DB
vectorstorev2 = Chroma(
    collection_name="summaries",
    embedding_function=embeddings
)

storev2 = InMemoryStore()
id_key = "doc_id"

retrieverv2 = MultiVectorRetriever(
    vectorstore=vectorstorev2,
    docstore=storev2,
    id_key=id_key
)

# Add docs
def add_documents_to_retriever(summaries, original_contents):
    doc_ids = [str(uuid.uuid4()) for _ in summaries]

    summary_docs = [
        Document(page_content=s, metadata={id_key: doc_ids[i]})
        for i, s in enumerate(summaries)
    ]

    retrieverv2.vectorstore.add_documents(summary_docs)
    retrieverv2.docstore.mset(list(zip(doc_ids, original_contents)))

C:\Users\sazza\AppData\Local\Temp\ipykernel_20672\3660238178.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4422.02it/s]
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## Now we can add everything to the Multivector Retriever
* for text and tables:
    * summaries are stored in the vector database.
    * raw documents are stored in the docstore.
* for the images:
    * summaries are stored in the vector database.
    * summaries (not the raw images) are also stored in the docstore.   

In [38]:
'''
# Add table summaries
add_documents_to_retriever(table_summaries, table_elements)
# Image
add_documents_to_retriever(image_summaries, image_paths)
'''

# Text
add_documents_to_retriever(text_summaries, texts)



## After adding that, we can now retrieve the information

In [ ]:
retrieverv2.invoke(
    "What do you see in the images?"
)

['The image features an abstract geometric design composed of two interlocking shapes. One shape is yellow and the other is dark blue. These shapes form a pattern that resembles an intertwined or connected structure, creating a sense of depth and movement. The design is symmetrical and dynamic.',
 'The image contains the word "STATEMENT" in bold, uppercase white letters against a dark blue background.',
 'The image contains the words "FINANCIAL STATEMENT." The text "FINANCIAL" is in large, bold, orange letters, while "STATEMENT" is in white. The background is dark blue.',
 'The image displays the text "$22,000,000 SALES" in bold, dark blue letters.']

#### Now, if we use the multi-vector retriever as the context:

In [ ]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

template = """Answer the question based only on the following context, which can include text, images and tables:
{context}
Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

chain = (
    {"context": retrieverv2, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

#### Then we can make questions about text, images or tables in the document

In [ ]:
chain.invoke(
     "What do you see on the images in the database?"
)

'The images in the database display information related to financial statements, sales figures, and return on investment (ROI). The text includes "$22,000,000 SALES," "STATEMENT," "FINANCIAL STATEMENT," and "33% ROI." The colors used are dark blue, white, and orange, with bold and uppercase letters for emphasis.'

In [ ]:
chain.invoke(
     "What is the name of the company?"
)

'The name of the company is not provided in the given context.'

In [ ]:
chain.invoke(
     "What is the product displayed in the image?"
)

'The product displayed in the image is a computer graphics card.'

In [ ]:
chain.invoke(
     "How much are the total expenses of the company?"
)

'The total expenses of the company are $2,000,000.'

In [ ]:
chain.invoke(
     "What is the ROI?"
)

'The ROI is 33%.'

In [ ]:
chain.invoke(
     "How much did the company sell in 2023?"
)

'The company sold approximately $18,000,000 in 2023.'

* Note: see that the previous answer can be seen as a mistake if we look at the bar chart, but we have to admit that the pdf is a bit confusing about it since it highlights de 22M sales in 2 different places.

In [ ]:
chain.invoke(
     "And in 2022?"
)

'In 2022, the data on the bar graph shows an approximate value of 15.'

* Note: see that now GPT4 is taking the right sales data from the bar chart. Impressive!

## How to execute the code from Visual Studio Code
* In Visual Studio Code, see the file 001-multimodal.py
* In terminal, make sure you are in the directory of the file and run:
    * python 001-multimodal.py